# Bradford Bulls — Sponsor Exposure (Colab GPU)

**Self-contained notebook — không cần upload src/, scripts/, hay bất kỳ thư mục nào.**
Chỉ cần upload video file là chạy được.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
!nvcc --version 2>/dev/null | grep release || echo 'nvcc not found'

## 2. Cài dependencies

CUDA toolkit 12.8 — dùng wheel `cu126` (PaddlePaddle chưa có cu128, cu126 chạy được nhờ backward compatibility).  
Warning về version mismatch là bình thường, không ảnh hưởng.

In [ ]:
!pip install -q "paddlepaddle-gpu==3.1.0" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr>=3.0" langchain-text-splitters langchain-community rapidfuzz ultralytics opencv-python-headless tqdm pandas matplotlib

In [ ]:
import paddle, torch
print('Paddle version :', paddle.__version__)
print('CUDA compiled  :', paddle.is_compiled_with_cuda())
print('GPU count      :', paddle.device.cuda.device_count())
print('Torch CUDA     :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Mount Google Drive + chọn video

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# >>> SỬA đường dẫn video trên Drive của bạn <<<
VIDEO_PATH = '/content/drive/MyDrive/bradford_bulls/videos/M05_white_1080p.mp4'

from pathlib import Path
print('Video:', VIDEO_PATH)
print('Exists:', Path(VIDEO_PATH).exists())

## 4. Cấu hình

In [ ]:
# ── Tham số chính ─────────────────────────────────────────────────────────────
SAMPLE_FPS    = 1.0     # frame/giây (1fps = production standard)
BOARDS        = True    # OCR full-frame cho biển quảng cáo pitch-side
MAX_SECONDS   = None    # None = full video; đặt số (vd 180) để test nhanh
MIN_PERSON_H  = 0.12    # bỏ qua người quá nhỏ (< 12% chiều cao frame)
GAP_MERGE_S   = 2.0     # gap (s) để ghép detection thành 1 lần xuất hiện

# OUTPUT
from pathlib import Path
stem     = Path(VIDEO_PATH).stem
OUT_DIR  = Path(f'exposure_output/{stem}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output → {OUT_DIR}/')

## 5. Code (tất cả inline — không cần file nào khác)

In [ ]:
# ── Sponsor dictionary + fuzzy matching ───────────────────────────────────────
# Thêm sponsor mới: 'ocr_token': 'CANONICAL_NAME'

from rapidfuzz import process, fuzz
import numpy as np

SPONSOR_DICT = {
    # Main jersey
    'topnotch':   'TOPNOTCH',    'top notch':   'TOPNOTCH',    'notch':      'TOPNOTCH',
    'floor':      'FLOOR TONIC', 'tonic':       'FLOOR TONIC', 'floortonic': 'FLOOR TONIC',
    'klg':        'KLG',         'klg europe':  'KLG',         'klgeurope':  'KLG',
    'aon':        'AON',
    'mcp':        'MCP',
    'fairway':    'FAIRWAY', 'roofing': 'FAIRWAY', 'ltd':        'FAIRWAY',
    'mna':        'MNA',
    'acs':        'ACS GROUP',   'acs group':   'ACS GROUP',   'acsgroup':   'ACS GROUP',
    # Shoulder / collar
    'mna':        'MNA',         'cladding':    'MNA',
    'bartercard': 'BARTERCARD',  'barte':       'BARTERCARD',
    'romantica':  'ROMANTICA',   'domantica':   'ROMANTICA',
    'chadwick':   'CHADWICK',    'chadlaw':     'CHADWICK',
    'atm':        'ATM',
    # Shorts
    'cch':        'CEDAR COURT', 'cedar court': 'CEDAR COURT', 'cedarcourt': 'CEDAR COURT', 'hotel': 'CEDAR COURT', 'cedar': 'CEDAR COURT',
    # Kit brand
    'ellgren':    'ELLGREN',
    'fourex':     'FOUREX',
    'paint': 'PAINT_AND_LACQUERS', 'paint and lacquers': 'PAINT_AND_LACQUERS', 'paintlacquers': 'PAINT_AND_LACQUERS', 'lacquers': 'PAINT_AND_LACQUERS', 'paintlacquer': 'PAINT_AND_LACQUERS',
}

MIN_FUZZY_SCORE = 80
MIN_OCR_CONF    = 0.55

_OCR_FIX      = str.maketrans({'0': 'o', '1': 'i', '|': 'i', '5': 's', '8': 'b'})
_DESPACED     = {k.replace(' ', ''): v for k, v in SPONSOR_DICT.items()}

def _normalize(text):
    return text.strip().lower().translate(_OCR_FIX)

def fuzzy_match_sponsor(text):
    t = _normalize(text)
    if len(t) < 2:
        return None
    best = None
    for table in (SPONSOR_DICT, _DESPACED):
        m = process.extractOne(t, table.keys(), scorer=fuzz.ratio,
                               score_cutoff=MIN_FUZZY_SCORE)
        if m and (best is None or m[1] > best[1]):
            best = (table[m[0]], int(m[1]))
    return best

print('Sponsor dict loaded:', len(SPONSOR_DICT), 'tokens →', len(set(SPONSOR_DICT.values())), 'sponsors')

In [ ]:
# ── Import PaddleOCR một lần ở module level (tránh paddlex init-lock khi re-run) ──
import paddle
import numpy as np
from paddleocr import PaddleOCR as _PaddleOCR

# ── Person detector (YOLO) ────────────────────────────────────────────────────
from ultralytics import YOLO
from dataclasses import dataclass
import cv2

class PersonDetector:
    def __init__(self, weights='yolov8n.pt', conf=0.4):
        self.model = YOLO(weights)
        self.conf  = conf

    def detect(self, frame_bgr, min_height_frac=0.12):
        H = frame_bgr.shape[0]
        results = self.model(frame_bgr, verbose=False, conf=self.conf, classes=[0])
        boxes = []
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                if (y2 - y1) / H >= min_height_frac:
                    boxes.append((x1, y1, x2, y2))
        return boxes


# ── OCR wrapper (PaddleOCR 3.x) ──────────────────────────────────────────────
@dataclass
class Detection:
    sponsor:  str
    text:     str
    conf:     float
    fuzzy:    int
    bbox:     tuple
    area_pct: float
    cx:       float
    cy:       float

def _expand_bbox(bbox, expand, frame_shape):
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    px, py = int(bw * expand), int(bh * expand)
    H, W   = frame_shape[:2]
    return max(0, x1-px), max(0, y1-py), min(W, x2+px), min(H, y2+py)

def _poly_to_bbox(poly):
    pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    x1, y1 = pts.min(axis=0).astype(int)
    x2, y2 = pts.max(axis=0).astype(int)
    return int(x1), int(y1), int(x2), int(y2)


class SponsorOCR:
    def __init__(self, lang='en', min_conf=MIN_OCR_CONF):
        device = 'gpu' if paddle.is_compiled_with_cuda() else 'cpu'
        self.ocr = _PaddleOCR(
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            lang=lang,
            device=device,
        )
        self.min_conf = min_conf
        print(f'SponsorOCR ready on {device.upper()}')

    def read(self, crop_bgr, frame_shape, offset=(0, 0),
             upscale_to=200, expand=0.15, matched_only=True):
        if crop_bgr.size == 0 or min(crop_bgr.shape[:2]) < 10:
            return []
        scale_up = max(1.0, upscale_to / crop_bgr.shape[0])
        if scale_up > 1.0:
            crop_bgr = cv2.resize(crop_bgr, None, fx=scale_up, fy=scale_up,
                                  interpolation=cv2.INTER_CUBIC)

        results = self.ocr.predict(crop_bgr)
        if not results:
            return []
        res    = results[0]
        polys  = res.get('rec_polys', [])
        texts  = res.get('rec_texts', [])
        scores = res.get('rec_scores', [])

        H, W = frame_shape[:2]
        frame_area = float(H * W)
        ox, oy = offset
        out = []

        for poly, text, conf in zip(polys, texts, scores):
            if conf < self.min_conf:
                continue
            match = fuzzy_match_sponsor(text)
            if match is None and matched_only:
                continue
            poly_arr  = np.asarray(poly, dtype=np.float32) / scale_up
            poly_arr += np.array([ox, oy], dtype=np.float32)
            bbox      = _expand_bbox(_poly_to_bbox(poly_arr), expand, frame_shape)
            x1, y1, x2, y2 = bbox
            area_pct  = (x2-x1)*(y2-y1) / frame_area * 100.0
            sponsor, fscore = match if match else ('', 0)
            out.append(Detection(
                sponsor=sponsor, text=text, conf=float(conf), fuzzy=fscore,
                bbox=bbox, area_pct=area_pct,
                cx=(x1+x2)/2/W, cy=(y1+y2)/2/H,
            ))
        return out

print('Classes defined: PersonDetector, SponsorOCR, Detection')

In [ ]:
# ── Aggregation ───────────────────────────────────────────────────────────────
from collections import defaultdict

def aggregate(per_frame, sample_dt, gap_merge_s=2.0):
    by_sponsor = defaultdict(list)
    for t, dets in per_frame.items():
        best = {}
        for d in dets:
            if d.sponsor not in best or d.area_pct > best[d.sponsor].area_pct:
                best[d.sponsor] = d
        for sp, d in best.items():
            by_sponsor[sp].append((t, d.area_pct, d.cx, d.cy, d.conf))

    rows = []
    for sponsor, recs in by_sponsor.items():
        recs.sort()
        times = [r[0] for r in recs]
        areas = [r[1] for r in recs]
        confs = [r[4] for r in recs]
        events = 1
        for i in range(1, len(times)):
            if times[i] - times[i-1] > gap_merge_s:
                events += 1
        rows.append(dict(
            sponsor       = sponsor,
            total_seconds = round(len(times) * sample_dt, 1),
            n_appearances = events,
            n_frames      = len(times),
            avg_area_pct  = round(float(np.mean(areas)), 3),
            peak_area_pct = round(float(np.max(areas)), 3),
            area_seconds  = round(sum(a * sample_dt for a in areas), 2),
            avg_ocr_conf  = round(float(np.mean(confs)), 3),
            first_seen_s  = round(min(times), 1),
            last_seen_s   = round(max(times), 1),
        ))
    rows.sort(key=lambda r: -r['area_seconds'])
    return rows

print('aggregate() ready')

## 6. Load detectors

In [ ]:
detector    = PersonDetector()   # auto-download yolov8n.pt nếu chưa có
sponsor_ocr = SponsorOCR()

## 7. Chạy đo exposure

In [ ]:
import cv2, csv
from tqdm.notebook import tqdm

cap       = cv2.VideoCapture(VIDEO_PATH)
fps_vid   = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration  = total_f / fps_vid

if MAX_SECONDS:
    total_f = min(total_f, int(MAX_SECONDS * fps_vid))

step      = max(1, int(round(fps_vid / SAMPLE_FPS)))
sample_dt = step / fps_vid
indices   = list(range(0, total_f, step))

print(f'Video    : {Path(VIDEO_PATH).name}')
print(f'Duration : {duration/60:.1f} min')
print(f'Sampling : {SAMPLE_FPS}fps → {len(indices)} frames (each = {sample_dt:.1f}s)')
print(f'Boards   : {"ON" if BOARDS else "OFF"}')

per_frame = {}
det_rows  = []

for idx in tqdm(indices, desc='measuring'):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if not ok:
        continue
    t = idx / fps_vid
    H, W = frame.shape[:2]
    frame_dets = []

    # jersey sponsors
    for (x1, y1, x2, y2) in detector.detect(frame, MIN_PERSON_H):
        bh  = y2 - y1
        ty1 = max(0, y1 + int(0.10 * bh))
        ty2 = min(H, y1 + int(0.75 * bh))
        torso = frame[ty1:ty2, x1:x2]
        frame_dets.extend(sponsor_ocr.read(torso, frame.shape, offset=(x1, ty1)))

    # pitch-side boards
    if BOARDS:
        frame_dets.extend(sponsor_ocr.read(frame, frame.shape, offset=(0, 0), upscale_to=H))

    if frame_dets:
        per_frame[t] = frame_dets
        for d in frame_dets:
            det_rows.append(dict(
                time_s=round(t, 2), sponsor=d.sponsor, text=d.text,
                ocr_conf=round(d.conf, 3), fuzzy=d.fuzzy,
                area_pct=round(d.area_pct, 3),
                cx=round(d.cx, 3), cy=round(d.cy, 3),
                bbox=','.join(map(str, d.bbox)),
            ))

cap.release()

# save detections.csv
det_csv = OUT_DIR / 'detections.csv'
with open(det_csv, 'w', newline='') as f:
    if det_rows:
        w = csv.DictWriter(f, fieldnames=list(det_rows[0].keys()))
        w.writeheader(); w.writerows(det_rows)
    else:
        f.write('no detections\n')

# aggregate
processed_s = len(indices) * sample_dt
report      = aggregate(per_frame, sample_dt, GAP_MERGE_S)
rep_csv     = OUT_DIR / 'exposure_report.csv'
with open(rep_csv, 'w', newline='') as f:
    if report:
        w = csv.DictWriter(f, fieldnames=list(report[0].keys()))
        w.writeheader(); w.writerows(report)
    else:
        f.write('no sponsors detected\n')

print(f'\nDone. {len(det_rows)} raw detections across {len(per_frame)} frames.')

## 8. Kết quả

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(rep_csv)
if 'sponsor' not in df.columns:
    print('Không có sponsor nào được detect.')
else:
    df['time_%'] = (df['total_seconds'] / processed_s * 100).round(1).astype(str) + '%'
    display(df[['sponsor','total_seconds','time_%','n_appearances',
                'avg_area_pct','peak_area_pct','area_seconds','avg_ocr_conf']])

In [ ]:
import matplotlib.pyplot as plt

r = df.sort_values('area_seconds', ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(r['sponsor'], r['area_seconds'], color='#c8102e')
axes[0].set_xlabel('Screen-area-seconds (AVE proxy)')
axes[0].set_title('Sponsor Exposure — AreaSec')

axes[1].barh(r['sponsor'], r['total_seconds'], color='#1a1a1a')
axes[1].set_xlabel('Time on screen (seconds)')
axes[1].set_title('Sponsor Exposure — Time')

plt.suptitle(f'Bradford Bulls — {stem}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'exposure_chart.png', dpi=150)
plt.show()
print(f'Chart saved → {OUT_DIR}/exposure_chart.png')

## 9. Audit — kiểm tra false positive

In [ ]:
import pandas as pd
from IPython.display import display

dets = pd.read_csv(det_csv)
if 'sponsor' not in dets.columns:
    print('Không có detection nào.')
else:
    print('Tổng detections:', len(dets))
    print('\nPhân bố theo sponsor:')
    print(dets['sponsor'].value_counts())
    print('\nConfidence thấp nhất (cần kiểm tra):')
    display(dets.sort_values('ocr_conf').head(10))

## 10. Visualize frames có sponsor detected

In [ ]:
import math, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import cv2

# ── Cấu hình visualize ────────────────────────────────────────────────────────
VIZ_MAX_FRAMES  = 24      # tối đa bao nhiêu frame hiển thị
VIZ_SPONSOR     = None    # None = tất cả; 'KLG' = chỉ frame có KLG
VIZ_RANDOM      = True    # True = random sample; False = N frame đầu tiên
VIZ_COLS        = 4       # số cột trong grid
VIZ_CONF_CUTOFF = 0.0     # chỉ hiện bbox có conf >= giá trị này

if not per_frame:
    print('Chưa có detection nào — chạy cell đo exposure trước.')
else:
    _PALETTE = [
        '#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
        '#42d4f4','#f032e6','#bfef45','#fabed4','#469990',
        '#dcbeff','#9A6324','#fffac8','#800000','#aaffc3',
    ]
    all_sponsors = sorted(set(d.sponsor for dets in per_frame.values() for d in dets))
    _COLOR = {sp: _PALETTE[i % len(_PALETTE)] for i, sp in enumerate(all_sponsors)}

    if VIZ_SPONSOR:
        candidate_times = sorted(
            t for t, dets in per_frame.items()
            if any(d.sponsor == VIZ_SPONSOR for d in dets)
        )
    else:
        candidate_times = sorted(per_frame.keys())

    if not candidate_times:
        print(f'Không tìm thấy frame nào có {VIZ_SPONSOR}.')
    else:
        if VIZ_RANDOM:
            show_times = sorted(random.sample(candidate_times,
                                              min(VIZ_MAX_FRAMES, len(candidate_times))))
        else:
            show_times = candidate_times[:VIZ_MAX_FRAMES]

        n_rows = math.ceil(len(show_times) / VIZ_COLS)
        fig, axes = plt.subplots(n_rows, VIZ_COLS,
                                 figsize=(VIZ_COLS * 4.5, n_rows * 3.2),
                                 squeeze=False)
        axes = axes.flatten()

        cap_viz = cv2.VideoCapture(VIDEO_PATH)
        for ax_i, t in enumerate(show_times):
            ax = axes[ax_i]
            cap_viz.set(cv2.CAP_PROP_POS_FRAMES, int(t * fps_vid))
            ok, frame = cap_viz.read()
            if not ok:
                ax.axis('off')
                continue
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            for d in per_frame[t]:
                if VIZ_SPONSOR and d.sponsor != VIZ_SPONSOR:
                    continue
                if d.conf < VIZ_CONF_CUTOFF:
                    continue
                x1, y1, x2, y2 = d.bbox
                color = _COLOR.get(d.sponsor, '#ffffff')
                ax.add_patch(patches.Rectangle(
                    (x1, y1), x2-x1, y2-y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                ))
                ax.text(x1, max(y1-4, 0), f'{d.sponsor}\n{d.text}  {d.conf:.2f}',
                        fontsize=6, color='white', fontweight='bold',
                        bbox=dict(facecolor=color, alpha=0.75, pad=1, edgecolor='none'))
            ax.set_title(f't={t:.0f}s', fontsize=8)
            ax.axis('off')
        cap_viz.release()

        for ax_i in range(len(show_times), len(axes)):
            axes[ax_i].axis('off')

        title = f'Detected frames — {stem}' + (f' ({VIZ_SPONSOR})' if VIZ_SPONSOR else '')
        plt.suptitle(title, fontsize=12, fontweight='bold')
        plt.tight_layout()
        viz_path = OUT_DIR / f'viz_{"all" if not VIZ_SPONSOR else VIZ_SPONSOR}.png'
        plt.savefig(viz_path, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'Saved → {viz_path}  ({len(show_times)}/{len(candidate_times)} frames)')

## 10. Download kết quả

In [ ]:
from google.colab import files
files.download(str(rep_csv))
files.download(str(det_csv))
files.download(str(OUT_DIR / 'exposure_chart.png'))

## 11. Export annotations → Roboflow (YOLO format)

Chạy OCR ở 2fps, lọc detection confidence cao, export ra:
```
roboflow_export/<stem>/
    images/   ← JPEGs
    labels/   ← YOLO .txt (class_id cx cy w h)
    classes.txt
    <stem>_roboflow.zip  ← upload thẳng lên Roboflow
```
Vào Roboflow: **New Project → Upload → chọn file zip → review → annotate tay chỗ thiếu**.

In [ ]:
import cv2, shutil
from pathlib import Path
from tqdm.notebook import tqdm

# ── Cấu hình export ───────────────────────────────────────────────────────────
EXPORT_FPS         = 1.0    # fps để sample
EXPORT_MIN_CONF    = 0.88   # OCR confidence tối thiểu
EXPORT_MIN_FUZZY   = 90     # fuzzy match score tối thiểu
EXPORT_MAX_SECONDS = None   # None = full video
EXPORT_BOARDS      = True   # bao gồm boards/biển quảng cáo

# Bbox expansion cho annotation (OCR bbox rất nhỏ — cần mở rộng để cover toàn logo)
# Jersey logo nhỏ → expand lớn; board logo to → expand nhỏ hơn
JERSEY_EXPAND  = 1.2   # mở rộng 120% mỗi chiều cho logo trên áo
BOARD_EXPAND   = 0.4   # mở rộng 40% mỗi chiều cho logo trên biển
MIN_BBOX_PX    = 60    # bbox tối thiểu 60px mỗi chiều (tránh annotation quá nhỏ)

# Output thẳng ra Google Drive
EXPORT_DIR = Path(f'/content/drive/MyDrive/bradford_bulls/roboflow_export/{stem}')

# ── Class map ─────────────────────────────────────────────────────────────────
all_classes = sorted(set(SPONSOR_DICT.values()))
class_to_id = {sp: i for i, sp in enumerate(all_classes)}

img_dir = EXPORT_DIR / 'images'
lbl_dir = EXPORT_DIR / 'labels'
img_dir.mkdir(parents=True, exist_ok=True)
lbl_dir.mkdir(parents=True, exist_ok=True)
(EXPORT_DIR / 'classes.txt').write_text('\n'.join(all_classes))
print('Classes:')
for i, c in enumerate(all_classes):
    print(f'  {i:2d}  {c}')

def expand_for_annotation(bbox, expand, min_px, frame_shape):
    """Mở rộng bbox OCR để cover toàn bộ vùng logo, không chỉ pixel chữ."""
    x1, y1, x2, y2 = bbox
    H, W = frame_shape[:2]
    bw, bh = x2 - x1, y2 - y1
    # expand theo tỷ lệ
    px = int(bw * expand)
    py = int(bh * expand)
    x1, y1, x2, y2 = x1 - px, y1 - py, x2 + px, y2 + py
    # enforce minimum size (center-anchored)
    if (x2 - x1) < min_px:
        cx = (x1 + x2) // 2
        x1, x2 = cx - min_px // 2, cx + min_px // 2
    if (y2 - y1) < min_px:
        cy = (y1 + y2) // 2
        y1, y2 = cy - min_px // 2, cy + min_px // 2
    # clamp to frame
    return (max(0, x1), max(0, y1), min(W, x2), min(H, y2))

# ── Sample + OCR + export ─────────────────────────────────────────────────────
cap     = cv2.VideoCapture(VIDEO_PATH)
fps_vid = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if EXPORT_MAX_SECONDS:
    total_f = min(total_f, int(EXPORT_MAX_SECONDS * fps_vid))

step    = max(1, int(round(fps_vid / EXPORT_FPS)))
indices = list(range(0, total_f, step))
print(f'\nSampling {EXPORT_FPS}fps → {len(indices)} frames...')

n_frames = n_annots = 0

for idx in tqdm(indices, desc='exporting'):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if not ok:
        continue

    H, W = frame.shape[:2]

    # jersey detections (label source = 'jersey')
    jersey_dets = []
    for (x1, y1, x2, y2) in detector.detect(frame, MIN_PERSON_H):
        bh  = y2 - y1
        ty1 = max(0, y1 + int(0.10 * bh))
        ty2 = min(H, y1 + int(0.75 * bh))
        torso = frame[ty1:ty2, x1:x2]
        dets = sponsor_ocr.read(torso, frame.shape, offset=(x1, ty1))
        jersey_dets.extend([(d, 'jersey') for d in dets])

    # boards detections
    board_dets = []
    if EXPORT_BOARDS:
        dets = sponsor_ocr.read(frame, frame.shape, offset=(0, 0), upscale_to=H)
        board_dets = [(d, 'board') for d in dets]

    # filter + deduplicate per sponsor (giữ conf cao nhất)
    best: dict = {}
    for d, source in jersey_dets + board_dets:
        if d.conf < EXPORT_MIN_CONF or d.fuzzy < EXPORT_MIN_FUZZY:
            continue
        key = d.sponsor
        if key not in best or d.conf > best[key][0].conf:
            best[key] = (d, source)

    if not best:
        continue

    # save JPEG
    frame_name = f'{stem}_{idx:07d}'
    cv2.imwrite(str(img_dir / f'{frame_name}.jpg'), frame,
                [cv2.IMWRITE_JPEG_QUALITY, 95])

    # save YOLO .txt — bbox mở rộng theo source
    lines = []
    for sp, (d, source) in best.items():
        expand = JERSEY_EXPAND if source == 'jersey' else BOARD_EXPAND
        bx1, by1, bx2, by2 = expand_for_annotation(
            d.bbox, expand, MIN_BBOX_PX, frame.shape)
        cx = (bx1 + bx2) / 2 / W
        cy = (by1 + by2) / 2 / H
        bw = (bx2 - bx1) / W
        bh = (by2 - by1) / H
        lines.append(f'{class_to_id[sp]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    (lbl_dir / f'{frame_name}.txt').write_text('\n'.join(lines))
    n_frames += 1
    n_annots += len(lines)

cap.release()

# zip
zip_path = EXPORT_DIR.parent / f'{stem}_roboflow'
shutil.make_archive(str(zip_path), 'zip', str(EXPORT_DIR.parent),
                    base_dir=EXPORT_DIR.name)

print(f'\n{"="*55}')
print(f'Export xong:')
print(f'  Frames      : {n_frames}')
print(f'  Annotations : {n_annots}')
print(f'  Zip         : {zip_path}.zip')
print(f'\nBước tiếp: Roboflow → New Project → Upload → chọn zip')